# Guesser prepared-data audit

This notebook audits the **production labels and split manifests** without a trained model. It checks the ordered label registry, per-class split counts, manifest row shape, and drawing-ID / trajectory-hash disjointness. The uniqueness check is disk-backed so it does not load all production IDs into notebook memory.

It does not calculate recognition metrics, inspect the indexed JSONL payload bytes, or render images. The data package's `guesser-data audit` checks the prepared payload; model predictions are required for accuracy or calibration. Keep the held-out test package sealed until the model and evaluation protocol are frozen.

Run from the repository root with `uv run --project models/guesser/evaluation jupyter lab models/guesser/evaluation/notebooks`. The default data path follows the repository layout; set `MTMAI_GUESSER_DATA_ROOT` to use another data root. Re-run the audit cell after the data pipeline finishes.

In [ ]:
# Reload our generated module after its writer cell runs, without restarting the kernel.
%load_ext autoreload
%autoreload 2

## Locate the prepared dataset

The variable may be overridden to point to an extracted trainer package. Do not point it at a held-out-test package before the candidate is frozen.

In [ ]:
# Resolve the helper path from this checkout so the notebook works from different cwd values.
from guesser_evaluation.notebook_tools import module_source_path

workflow_source_path = str(module_source_path('dataset_audit_workflow.py'))
print(f'Workflow source: {workflow_source_path}')

In [ ]:
%%writefile {workflow_source_path}
"""Notebook helpers for locating, auditing, and plotting Guesser data."""

from __future__ import annotations

import os
from pathlib import Path
from typing import Any


def production_data_root() -> Path:
    """Resolve the default production data root or the portable override."""
    from .notebook_tools import module_source_path

    # Infer the sibling data project from this source file, not a machine-specific path.
    source_path = module_source_path("dataset_audit_workflow.py")
    default_root = source_path.parents[2].parent / "data" / ".local" / "data" / "production"
    # Let another clone or extracted package override the repository's local default.
    configured_root = os.environ.get("MTMAI_GUESSER_DATA_ROOT", str(default_root))
    return Path(configured_root).expanduser()


def audit_production_dataset(data_root: str | Path | None = None) -> dict[str, Any] | None:
    """Audit prepared production manifests, or return None until they exist."""
    from .dataset_audit import audit_prepared_dataset

    # Use the explicit argument when supplied; otherwise resolve the portable default.
    root = Path(data_root).expanduser() if data_root is not None else production_data_root()
    prepared_path = root / "prepared"
    # A missing folder means preparation is pending, not that the audit passed.
    if not prepared_path.is_dir():
        return None
    return audit_prepared_dataset(
        prepared_path,
        # Enforce the production label set and per-class split contract.
        expected_class_count=345,
        expected_per_class={"train": 4000, "validation": 500, "test": 500},
    )


def plot_class_balance(report: dict[str, Any]):
    """Return per-split totals and a class-count figure from a passed audit."""
    import matplotlib.pyplot as plt
    import pandas as pd

    counts = pd.DataFrame(report["counts_by_class"])
    # Sum the same audited rows shown in the plot for a quick total check.
    totals = counts[["train", "validation", "test"]].sum()
    # Stack partitions so class balance can be compared without hiding any split.
    figure, axis = plt.subplots(figsize=(18, 5))
    counts.set_index("source_name")[["train", "validation", "test"]].plot(
        kind="bar", stacked=True, width=0.9, ax=axis
    )
    axis.set_title("Original drawings by Quick, Draw! class and split")
    axis.set_xlabel("Class")
    axis.set_ylabel("Original drawings")
    figure.tight_layout()
    return totals, figure


## Audit labels, counts, and split isolation

Production expects all 345 labels and 4,000 / 500 / 500 originals per class in train / validation / test. The function reads Parquet manifests in batches and uses a temporary SQLite index to detect duplicate identities and trajectories across all splits. It does not open the test package unless the selected data root points to it.

In [ ]:
# Refresh discovery after %%writefile creates the reusable workflow module.
import importlib
importlib.invalidate_caches()
from guesser_evaluation.dataset_audit_workflow import audit_production_dataset

audit_report = audit_production_dataset()
if audit_report is None:
    # Missing data means acquisition/preparation is pending, not that validation passed.
    audit_report = None
    print('Prepared data is not available yet. Re-run this cell after download and prepare finish.')
audit_report

## Inspect class balance

The chart visualizes counts from the manifests after the audit passes. It is a dataset-structure check, not model performance. A failed audit raises an error above instead of silently omitting rows.

In [ ]:
from IPython.display import display

if audit_report is not None:
    # Only visualize class balance after every manifest check has passed.
    from guesser_evaluation.dataset_audit_workflow import plot_class_balance

    split_totals, class_balance_figure = plot_class_balance(audit_report)
    display(split_totals.to_frame(name='originals'))
    class_balance_figure
else:
    print('No prepared manifests to summarize yet.')

## Limits and next step

A passing result establishes that the label/split manifests satisfy these checks; it does **not** establish that each indexed JSONL byte range is intact or that preprocessing looks correct. Run the data package audit for payload integrity. When checkpoints arrive, extend evaluation with the fixed ten-stage protocol, keeping blank/pass opportunities in the denominator and using validation for model selection and calibration. Do not tune against held-out test results.

The writer cell is the source of truth for `src/guesser_evaluation/dataset_audit_workflow.py`; execute it to regenerate the importable module. The independent manifest auditor lives in `src/guesser_evaluation/dataset_audit.py`. Keep both modules covered by tests.